In [ ]:
import os

print("🔍 Checking mounted input folders...")

# Confirm what's mounted at /kaggle/input
input_folders = sorted(os.listdir('/kaggle/input'))
print("\nTop-level folders in /kaggle/input:")
for f in input_folders:
    print("  -", f)

# Check test planet subfolders
test_path = '/kaggle/input/ariel-data-challenge-2025/test'
if os.path.exists(test_path):
    test_subdirs = sorted(os.listdir(test_path))
    print(f"\nSubfolders in '{test_path}':")
    for tid in test_subdirs[:10]:  # show first 10
        print("  -", tid)
    print(f"... ({len(test_subdirs)} total)")
else:
    print(f"\n🚫 Path does not exist: {test_path}")


In [ ]:
## Cell 1

import os
import pandas as pd
import numpy as np

print("PHOENIX V20 FOUNDATION VALIDATION")
print("=" * 50)

print("Available input directories:")
for item in os.listdir('/kaggle/input'):
    print(f"  {item}")

In [ ]:
## Cell 2
import pandas as pd
import os

test_root = '/kaggle/input/ariel-data-challenge-2025/test'
example_id = '1103775'  # or whichever test planet you want to analyze

signal_path = os.path.join(test_root, example_id, 'AIRS-CH0_signal_0.parquet')
airs_data = pd.read_parquet(signal_path)

print(f"Loaded spectrum for test ID {example_id}, shape: {airs_data.shape}")


In [ ]:
## Cell 3

import os
import pandas as pd
import numpy as np

def extract_transit_spectrum(airs_data, transit_start=5200, transit_end=6600):
    transit_depths = []
    uncertainties = []

    # Process in chunks for speed (optional)
    for i in range(0, airs_data.shape[1], 1000):
        for pixel_idx in range(i, min(i+1000, airs_data.shape[1])):
            pixel_name = f'column_{pixel_idx}'
            pixel_data = airs_data[pixel_name].values

            # Baseline flux (out of transit)
            out_flux_1 = pixel_data[1000:3000].mean()
            out_flux_2 = pixel_data[8000:10000].mean()
            baseline_flux = (out_flux_1 + out_flux_2) / 2

            # Transit flux (minimum during transit)
            transit_flux = pixel_data[transit_start:transit_end].min()

            # Transit depth (normalized)
            depth = (baseline_flux - transit_flux) / baseline_flux

            # Simple uncertainty estimate
            baseline_std = np.std(np.concatenate([pixel_data[1000:3000], pixel_data[8000:10000]]))
            uncertainty = baseline_std / baseline_flux

            transit_depths.append(depth)
            uncertainties.append(uncertainty)

    return np.array(transit_depths), np.array(uncertainties)


test_root = '/kaggle/input/ariel-data-challenge-2025/test'
test_ids = sorted(os.listdir(test_root))

test_depths = []
test_uncertainties = []

for tid in test_ids:
    signal_path = os.path.join(test_root, tid, 'AIRS-CH0_signal_0.parquet')
    airs_data = pd.read_parquet(signal_path)
    
    depths, uncerts = extract_transit_spectrum(airs_data)
    test_depths.append(depths)
    test_uncertainties.append(uncerts)

print("Extraction complete.")
print(f"Number of test spectra processed: {len(test_ids)}")
print("Example pixel depths:", test_depths[0][:10])
print("Example uncertainties:", test_uncertainties[0][:10])



In [ ]:
## Cell 4

## Cell 4 (Improved Extraction for All Test IDs)

print("IMPROVED SPECTRAL EXTRACTION FOR ALL TEST CASES")
print("-" * 50)

transit_start = 5200
transit_end = 6600

improved_depths_all = []
improved_uncertainties_all = []

for tid in test_ids:
    signal_path = os.path.join(test_root, tid, 'AIRS-CH0_signal_0.parquet')
    airs_data = pd.read_parquet(signal_path)

    improved_depths = []
    improved_uncertainties = []

    for pixel_idx in range(0, min(1000, airs_data.shape[1])):
        pixel_name = f'column_{pixel_idx}'
        pixel_data = airs_data[pixel_name].values

        out_flux_1 = np.median(pixel_data[1000:3000])
        out_flux_2 = np.median(pixel_data[8000:10000])
        baseline_flux = (out_flux_1 + out_flux_2) / 2

        transit_flux = np.median(pixel_data[transit_start:transit_end])
        depth = (baseline_flux - transit_flux) / baseline_flux

        baseline_combined = np.concatenate([pixel_data[1000:3000], pixel_data[8000:10000]])
        baseline_rms = np.std(baseline_combined)
        photon_noise = np.sqrt(baseline_flux) / baseline_flux
        systematic_noise = baseline_rms / baseline_flux
        total_uncertainty = np.sqrt(photon_noise**2 + systematic_noise**2)

        if 0.001 < depth < 0.5 and total_uncertainty < 0.1:
            improved_depths.append(depth)
            improved_uncertainties.append(total_uncertainty)

    improved_depths_all.append(improved_depths)
    improved_uncertainties_all.append(improved_uncertainties)

    print(f"{tid}: Good pixels = {len(improved_depths)}/1000 | "
          f"Depth range = {np.min(improved_depths)*100:.2f}%–{np.max(improved_depths)*100:.2f}% | "
          f"Mean unc = {np.mean(improved_uncertainties)*100:.3f}%")


In [ ]:
## Cell 4a

def weighted_depth(depths, uncertainties):
    """
    Computes a weighted average of transit depths using inverse-variance weighting.
    Returns a single scalar prediction.
    """
    weights = 1.0 / (uncertainties**2 + 1e-8)  # Prevent div by zero
    weighted_avg = np.sum(depths * weights) / np.sum(weights)
    return weighted_avg


submission_rows = []

for i in range(len(test_ids)):
    depths = np.array(improved_depths_all[i])
    uncertainties = np.array(improved_uncertainties_all[i])
    pred = weighted_depth(depths, uncertainties)
    submission_rows.append((test_ids[i], pred))


# Construct the submission DataFrame
submission = pd.DataFrame(submission_rows, columns=["ID", "transit_depth"])

# Validation checks
print(f"Prediction loop complete: {len(submission_rows)} entries created.")
print("Sample submission rows:", submission_rows[:3])
print("Submission columns:", submission.columns.tolist())
print("Submission shape:", submission.shape)
print("Null values per column:\n", submission.isnull().sum())
print("Any infinite values in 'transit_depth':", (~np.isfinite(submission['transit_depth'])).any())
print("Sample submission rows:\n", submission.head())

# Save to file
submission.to_csv('submission.csv', index=False)
print("✅ Submission file saved as 'submission.csv'")

In [ ]:
## Cell 4b

submission_rows = []

for i in range(len(test_ids)):
    pred = weighted_depth(improved_depths[i], improved_uncertainties[i])
    submission_rows.append((test_ids[i], pred))

print(f"Prediction loop complete: {len(submission_rows)} entries created.")
print("Sample submission rows:", submission_rows[:3])

submission = pd.DataFrame(submission_rows, columns=["ID", "transit_depth"])

print("Submission columns:", submission.columns.tolist())
print("Submission shape:", submission.shape)
print("Null values per column:\n", submission.isnull().sum())
print("Any infinite values in `transit_depth`:", (~np.isfinite(submission['transit_depth'])).any())
print("Sample submission rows:\n", submission.head())

submission.to_csv('submission.csv', index=False)
print("Submission file saved as 'submission.csv'")


In [ ]:
## Cell 5

print("FINDING PRECISE TRANSIT TIMING:")
print("-" * 40)

# Find the exact transit duration by looking at flux evolution
# Average across many pixels to reduce noise
mean_flux_evolution = airs_data[['column_1000', 'column_2000', 'column_3000', 'column_4000', 'column_5000']].mean(axis=1)

# Look at the flux in our suspected transit window
transit_window = mean_flux_evolution[transit_start:transit_end]
window_steps = np.arange(transit_start, transit_end)

# Find the baseline within this window - CORRECTED
baseline_segment_1 = mean_flux_evolution[1000:3000]
baseline_segment_2 = mean_flux_evolution[8000:10000]
baseline_combined = np.concatenate([baseline_segment_1, baseline_segment_2])
baseline_level = np.median(baseline_combined)

# Find points significantly below baseline (actual transit)
transit_threshold = baseline_level * 0.98  # 2% depth threshold
in_transit_mask = transit_window < transit_threshold

# Find continuous transit period
in_transit_indices = window_steps[in_transit_mask]

if len(in_transit_indices) > 0:
    true_transit_start = in_transit_indices[0]
    true_transit_end = in_transit_indices[-1]
    transit_duration = true_transit_end - true_transit_start
    
    print(f"Precise transit timing:")
    print(f"  Start: step {true_transit_start}")
    print(f"  End: step {true_transit_end}")
    print(f"  Duration: {transit_duration} steps")
    print(f"  Fraction of orbit: {transit_duration/11250*100:.1f}%")
    
    # Show the difference
    precise_transit_depth = (baseline_level - mean_flux_evolution[true_transit_start:true_transit_end].min()) / baseline_level
    window_median_depth = (baseline_level - transit_window.median()) / baseline_level
    
    print(f"\nDepth comparison:")
    print(f"  Using precise timing: {precise_transit_depth*100:.2f}%")
    print(f"  Using window median: {window_median_depth*100:.2f}%")
else:
    print("No clear transit found - need to adjust threshold")

In [ ]:
import os
print(sorted(os.listdir('/kaggle/input/ariel-data-challenge-2025/test')))


In [ ]:
## Cell 6

## Cell 6: Full-batch spectrum extraction using precise transit timing

import os
import pandas as pd
import numpy as np

test_root = '/kaggle/input/ariel-data-challenge-2025/test'
test_ids = sorted(os.listdir(test_root))

# Precise transit window
precise_start = 5298
precise_end = 5398

# Results
corrected_depths_all = []
corrected_uncertainties_all = []

print("Starting full-batch processing...")

for tid in test_ids:
    signal_path = os.path.join(test_root, tid, 'AIRS-CH0_signal_0.parquet')
    airs_data = pd.read_parquet(signal_path)

    depths = []
    uncertainties = []

    for pixel_idx in range(airs_data.shape[1]):
        pixel_name = f'column_{pixel_idx}'
        pixel_data = airs_data[pixel_name].values

        baseline = np.concatenate([pixel_data[1000:3000], pixel_data[8000:10000]])
        baseline_flux = np.median(baseline)

        transit_flux = pixel_data[precise_start:precise_end].min()

        depth = (baseline_flux - transit_flux) / baseline_flux

        baseline_rms = np.std(baseline)
        photon_noise = np.sqrt(baseline_flux) / baseline_flux
        systematic_noise = baseline_rms / baseline_flux
        total_unc = np.sqrt(photon_noise**2 + systematic_noise**2)

        if 0.001 < depth < 0.5 and total_unc < 0.1:
            depths.append(depth)
            uncertainties.append(total_unc)

    corrected_depths_all.append(np.array(depths))
    corrected_uncertainties_all.append(np.array(uncertainties))

print("✅ Full-batch extraction complete.")



In [ ]:
## Cell 7

## Cell 7: Weighted prediction logic

def weighted_depth(corrected_depths, corrected_uncertainties,
                   depth_min=0.005, depth_max=0.08, uncertainty_max=0.08):
    realistic_depths = []
    pixel_weights = []
    for d, u in zip(corrected_depths, corrected_uncertainties):
        if (depth_min < d < depth_max and u < uncertainty_max):
            snr = d / u
            pixel_weights.append(max(snr, 0.01))  # avoid zero weights
            realistic_depths.append(d)

    if not realistic_depths:
        return float(np.mean(corrected_depths))  # fallback
    realistic_depths = np.array(realistic_depths)
    pixel_weights = np.array(pixel_weights)
    return float(np.average(realistic_depths, weights=pixel_weights))



In [ ]:
## Cell 8

## Cell 8: Generate final predictions

submission_rows = []

for i in range(len(test_ids)):
    pred = weighted_depth(corrected_depths_all[i], corrected_uncertainties_all[i])
    submission_rows.append((test_ids[i], pred))

print(f"Prediction loop complete: {len(submission_rows)} entries created.")
print("Sample predictions:", submission_rows[:3])



In [ ]:
## Cell 9

## Cell 9: Create and validate submission

submission = pd.DataFrame(submission_rows, columns=["ID", "transit_depth"])

print("Submission columns:", submission.columns.tolist())
print("Submission shape:", submission.shape)
print("Nulls per column:\n", submission.isnull().sum())
print("Infs per column:\n", (~np.isfinite(submission['transit_depth'])).sum())
print("Sample:\n", submission.head())

submission.to_csv("submission.csv", index=False)
print("✅ Submission file saved as `submission.csv`")



In [ ]:
## Cell 9a

submission_rows = []

for i in range(len(test_ids)):
    pred = weighted_depth(test_depths[i], test_uncertainties[i])
    submission_rows.append((test_ids[i], pred))

print(f"Prediction loop complete: {len(submission_rows)} entries created.")
print("Sample submission rows:", submission_rows[:3])


In [ ]:
## Cell 10

# Create submission DataFrame from prediction results
submission = pd.DataFrame(submission_rows, columns=["ID", "transit_depth"])

# Validate submission before saving
print("Submission columns:", submission.columns.tolist())
print("Submission shape:", submission.shape)
print("Null values per column:\n", submission.isnull().sum())
print("Any infinite values in 'transit_depth':", ~np.isfinite(submission['transit_depth']).all())
print("Sample submission rows:\n", submission.head())

# Save submission CSV file
submission.to_csv('submission.csv', index=False)
print("Submission file saved as 'submission.csv'")


In [ ]:
# Cell 11: Submission sanity check

import matplotlib.pyplot as plt
import numpy as np

print("Quick submission summary:")
print("Shape:", submission.shape)
print("First few rows:\n", submission.head())

plt.hist(submission['transit_depth'], bins=50)
plt.xlabel("Predicted Transit Depth")
plt.ylabel("Count")
plt.title("Submission Distribution")
plt.show()

# Check for bad values
print("NaNs in submission:", submission['transit_depth'].isna().sum())
print("Infs in submission:", (~np.isfinite(submission['transit_depth'])).sum())


In [ ]:
# Cell 12: Fallback usage diagnostics

num_fallbacks = 0
for i in range(len(test_ids)):
    # Count if all pixels in a spectrum fail the physical cuts
    use_fallback = not any(
        (0.005 < d < 0.08 and u < 0.08)
        for d, u in zip(test_depths[i], test_uncertainties[i])
    )
    if use_fallback:
        num_fallbacks += 1

print(f"Number of fallback predictions used: {num_fallbacks} out of {len(test_ids)}")
if num_fallbacks > 0:
    print("Consider investigating spectra with all pixels outside physical range.")
